In [1]:
import cv2
import mediapipe as mp
import numpy as np
from deepface import DeepFace
import socket
import pickle
import time
#logic to create socket connection with unity
soc = socket.socket()
hostname = "localhost"
port = 5000
soc.bind((hostname, port))
soc.listen(5)
conn, addr = soc.accept()
print("Device Connected")
old_msg=''
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.65,#used to be 0.5, increased for better accuracy
    model_complexity=1
)

face_ids = {}
frame_count = 0
cap = cv2.VideoCapture("http://192.168.1.100:8080/video")

while cap.isOpened():
    _, frame = cap.read()
    msg=''
    try:
        
        frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        f_frame = cv2.resize(frame, (480, 640))
        frame_rgb = cv2.cvtColor(f_frame, cv2.COLOR_BGR2RGB)
        frame_count += 1
        if frame_count % 60 == 0:
            face_encodings = DeepFace.represent(
                f_frame,
                model_name="Facenet",
                enforce_detection=False
            )
            if len(face_encodings) > 0:
                face_id = tuple(face_encodings[0]["embedding"])
                match = None
                for k in face_ids:
                    if np.linalg.norm(np.array(face_id) - np.array(k)) < 15:
                        match = face_ids[k]
                        msg = "Known face recognized: " + match
                        break
                if match is None:
                    face_ids[face_id] = "Person " + str(len(face_ids) + 1)
                    msg = "New face detected: " + face_ids[face_id]
                    print("New face detected: " + face_ids[face_id])
                else:
                    print("Known face recognized: " + match)
        results = holistic.process(frame_rgb)
        annotated_image = f_frame.copy()
        for face_id_key, name in face_ids.items():
            cv2.putText(annotated_image, name, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        mp_drawing.draw_landmarks(annotated_image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(annotated_image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(
            annotated_image,
            results.face_landmarks,
            mp_holistic.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )
        mp_drawing.draw_landmarks(
            annotated_image,
            results.pose_landmarks,
            mp_holistic.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
        )
        cv2.imshow("Output", annotated_image)
        #logic to send msg to unity
        if msg!='' and msg != old_msg:  # only send when there's actually something
            conn.send(msg.encode('utf-8'))
        old_msg = msg

        if msg == pickle.dumps("exit"):
            break
    except Exception as e:
        print(e)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf
print(tf.__version__)